<!-- music_separater.ipynb: GPU 付き Colab で音源分離アプリを使うための共有用入口。 -->
# 🎵 Music Separater — 音楽をボーカル・伴奏に分けよう

**コードを書く必要はありません。①と②の左にある ▶ を順番に押し、プログラムを実行してください。**

1. このノートブックを Google Colab で開き、「ファイル → ドライブにコピーを保存」で自分用のコピーを作ります。
2. 上の「ランタイム → ランタイムのタイプを変更」で **Python 3 / GPU / T4** を選んで保存します。
3. ①で準備ができたら、②を実行すると、このページ内に操作画面が表示されます。

Colab の Python は変更不要です。①がアプリ専用の Python 3.12 と GPU 用パッケージを準備します。

初回の準備と最初の分離には、パッケージ・モデルの取得で時間がかかります。完了まで Colab を開いておいてください。


In [ ]:
# @title ① 準備する（最初に一度だけ ▶）
# @markdown 大半の場合、すでに入っているurlから変更する必要はなく、revisionも空白のままで大丈夫です
# Colab の標準ライブラリだけで取得し、アプリ用の準備処理へ渡します。
repository_url = "https://github.com/sorawww31/musicseparater.git" # @param {type:"string"}

#空欄で大丈夫
revision = "" # @param {type:"string"}

import json
import os
import re
import subprocess
import sys
from pathlib import Path

if "google.colab" not in sys.modules:
    raise RuntimeError("このノートブックを Google Colab で開いてください。")
repository_url, revision = repository_url.strip(), revision.strip()
if not re.fullmatch(r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", repository_url):
    raise ValueError("repository_url に共有者から案内された https://github.com/所有者/リポジトリ を入力してください。")
if revision.startswith("-"):
    raise ValueError("revision はブランチ名・タグ名・コミット ID を入力してください。")
repository = Path("/content/musicseparater")
source = {"url": repository_url, "revision": revision}
marker = repository / ".colab/source.json"
if repository.exists():
    if not (repository / ".git").is_dir():
        raise RuntimeError("取得先に別のフォルダーがあります。ランタイムを削除して①からやり直してください。")
    origin = subprocess.check_output(["git", "-C", str(repository), "remote", "get-url", "origin"], text=True).strip()
    if origin != repository_url or (marker.exists() and json.loads(marker.read_text()) != source):
        raise RuntimeError("取得先やバージョンを変える場合は、ランタイムを削除して①からやり直してください。")
else:
    subprocess.run(["git", "clone", "--depth=1", repository_url, str(repository)], check=True, env=dict(os.environ, GIT_TERMINAL_PROMPT="0"))
if revision and not marker.exists():
    subprocess.run(["git", "-C", str(repository), "fetch", "--depth=1", "origin", revision], check=True)
    subprocess.run(["git", "-C", str(repository), "checkout", "--detach", "FETCH_HEAD"], check=True)
if not (repository / "colab_runtime/setup.py").is_file():
    raise RuntimeError("取得先に Colab 対応コードがありません。共有者に URL と revision を確認してください。")
marker.parent.mkdir(exist_ok=True)
marker.write_text(json.dumps(source))
if str(repository) not in sys.path:
    sys.path.insert(0, str(repository))
from colab_runtime import launcher
launcher.stop()
# 子プロセスの出力を Python 経由でセルへ転送し、失敗理由も表示します。
with subprocess.Popen(
    [sys.executable, "-u", "-m", "colab_runtime.setup"], cwd=repository,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    encoding="utf-8", errors="replace",
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("準備に失敗しました。このメッセージの直前に表示されたログを確認してください。")

## ② 操作画面を開く

下の ▶ を押し、表示された画面で操作します。**途中で画面を閉じても、②を再実行すれば開けます。**

1. 「対象楽曲」に **MP3 / WAV / FLAC** を選びます。
2. はじめは **2ステム → BS PolarFormer** のままで大丈夫です。
3. 「分離を開始」を押し、「完了」まで待ちます。初回はモデル取得中に進捗が止まって見えることがあります。
4. **Vocals = 歌声 / Instrumental = 伴奏** です。再生して確認し、「WAVをダウンロード」で保存します。

2 人の歌声を分けたいときは「複数歌声 → Blind separation → UNMIXX」、特定の歌手を取り出す場合は「Target singer extraction」を選び、その歌手だけが歌う 3 秒以上の参照音声も追加します。

**SepACap は BF16 対応 GPU が必要です。T4 では他のモデルを選んでください。**


In [ ]:
# @title ② アプリを開く ▶
# Colab 公式の iframe 表示を使い、音声と API を同じポートで扱います。
from pathlib import Path
import sys
repository = Path("/content/musicseparater")
if not (repository / ".colab/ready").exists():
    raise RuntimeError("先に「① 準備する」を最後まで実行してください。")
if str(repository) not in sys.path:
    sys.path.insert(0, str(repository))
from google.colab import output
from colab_runtime import launcher
from colab_runtime.config import IFRAME_HEIGHT
port = launcher.start()
output.serve_kernel_port_as_iframe(port, height=IFRAME_HEIGHT, cache_in_notebook=False)


## 保存・終了・困ったとき

- **終了前に WAV をダウンロードしてください。** 音声・分離結果・モデルは Colab の一時ディスクに保存されます。ランタイムが削除されると消えます。Google Drive への自動保存はありません。
- 友人にはこの **ノートブック** を共有してください。それぞれが自分の Colab と GPU で実行します。画面の URL を渡す方式ではありません。
- GPU がないと表示されたら、ランタイムの設定を GPU に変更してください。GPU を割り当てられない場合は、時間を置いて再試行してください。
- 画面が出ない場合は①の完了を確認し、②をもう一度実行してください。
- 分離に失敗したら画面のエラーと下のログを確認してください。GPU メモリ不足なら短い音源で試し、必要ならランタイムを再起動して①から実行します。
- 再起動や停止で処理が中断した音源は、最初から分離し直してください。
- 共有時は「編集 → すべての出力を消去」して保存してください。ログや実行結果を友人に渡さずに済みます。
- 各モデルの用途・利用条件はリポジトリの `backend/README.md` を確認してください。

通常は以下のセルを実行する必要はありません。


In [ ]:
# @title 困ったとき：ログを表示する ▶
# 実行結果を共有する際はファイル名などを確認してください。
from colab_runtime.launcher import show_log
show_log()


In [ ]:
# @title 終了するとき：アプリを停止する（チェックして ▶）
# 「すべてのセルを実行」しても直ちに終了しないよう、明示的に選択します。
stop_application = False # @param {type:"boolean"}
if stop_application:
    from colab_runtime.launcher import stop
    stop()
    print("停止しました。GPU の割り当ても終了するには「ランタイム → 接続を解除してランタイムを削除」を選んでください。")
else:
    print("アプリはそのまま利用できます。終了時だけチェックして実行してください。")
